# Mapping with Predefined Lists

In [28]:
import os, sys
import findspark
findspark.init()
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import List as mapping_list

In [29]:
import importlib
importlib.reload(mapping_list)

<module 'List' from '/mnt/01DBA8B8279979A0/Pulse - E-Commerce Data Analytics Engine/mapping/List.py'>

In [30]:
def normalize_dataframe(df, column_variants):
    variant_to_standard = {
        v.lower(): std_col
        for std_col, variants in column_variants.items()
        for v in variants
    }

    mapped_cols = {}
    new_columns = []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in variant_to_standard:
            std_col = variant_to_standard[col_lower]
            new_columns.append(std_col)
            mapped_cols[std_col] = col
        else:
            new_columns.append(col)

    for old_col, new_col in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old_col, new_col)

    missing_cols = []
    for std_col in column_variants.keys():
        if std_col not in df.columns:
            df = df.withColumn(std_col, lit(None))
            missing_cols.append(std_col)

    schema_cols = list(column_variants.keys())
    extra_cols = [c for c in df.columns if c not in schema_cols]

    new_df = df.select(schema_cols)
    df_extra = df.select(schema_cols + extra_cols)

    return new_df, df_extra, extra_cols, missing_cols, mapped_cols

In [31]:
base_dir = os.getcwd()
file_path_excel = os.path.join(base_dir, "./../faker/messy_inventory_data.xlsx")
file_path_csv = os.path.join(base_dir, "./../faker/messy_inventory_data.csv")

In [32]:
excel_df = pd.read_excel(file_path_excel, engine="openpyxl")
excel_df.to_csv(file_path_csv, index=False)

In [33]:
spark = SparkSession.builder.appName("NormalizeData").getOrCreate()
df = spark.read.csv(file_path_csv, header=True, inferSchema=True)

In [34]:
df.show(5)

+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inv_id|product_ref|   vendor_id|current_stock|reserved_stock|min_stock_level|   last_restock_date|      last_sale_date|monthly_storage_cost|        created_date|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------+-----------+------------+-------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|100555|  PROD_0781|    SUPP_017|           72|            21|             69|2025-08-25 18:33:...|2025-09-07 03:54:...|                0.74|2024-05-31

In [35]:
new_df, extra_df, extra_cols, missing, mapped = normalize_dataframe(
    df, mapping_list.mapping_dict_inventory
)

In [36]:
print("\nNormalized DataFrame:")
new_df.show(5)

print("\nDataFrame with Extra Columns:")
extra_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)


Normalized DataFrame:
+------------+----------+------------+--------------+-----------------+-------------+-------------------+--------------+------------+----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_quantity|reorder_level|last_restocked_date|last_sold_date|storage_cost|created_at|
+------------+----------+------------+--------------+-----------------+-------------+-------------------+--------------+------------+----------+
|      100555| PROD_0781|    SUPP_017|            72|             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100938| PROD_0373|    SUPP_003|             8|             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100060| PROD_9999|  SUPP_029  |        Many  |             NULL|         NULL|               NULL|          NULL|        NULL|      NULL|
|      100693| PROD_0244|    SUPP_022|         240  |             NULL|         NULL|               NULL|  

# Implementing RapidFuzz

In [37]:
from rapidfuzz import fuzz, process


def rapidfuzz_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=85):
    """
    Map columns using RapidFuzz's string matching algorithms
    Returns normalized dataframe, remaining missing columns, extra columns, and mapped columns
    """
    for missing_col in missing_cols[:]:
        # Use process.extractOne to find the best match
        match = process.extractOne(
            missing_col,
            extra_cols,
            scorer=fuzz.ratio,  # Can also use fuzz.WRatio or fuzz.token_sort_ratio
            score_cutoff=threshold,
        )

        if match:  # match will be (matched_string, score, index)
            best_match, score = match[0], match[1]
            print(f"Mapping: {best_match} -> {missing_col}: {score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    # Rename columns based on mapping
    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols


# Example usage:
new_df, missing, extra_cols, mapped = rapidfuzz_column_mapping(
    df, missing, extra_cols, mapped, threshold=85
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

Mapping: last_restock_date -> last_restocked_date: 94.44
Mapping: last_sale_date -> last_sold_date: 85.71
Mapping: created_date -> created_at: 90.91

Normalized DataFrame:
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_stock|min_stock_level| last_restocked_date|      last_sold_date|monthly_storage_cost|          created_at|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+------------+------------------+---------------

# Implementing NLTK

In [38]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.metrics.distance import edit_distance
from difflib import SequenceMatcher
import re

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/khalid_ah_1/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [39]:
def preprocess_column_name(column):
    """Split camelCase and snake_case, convert to lowercase"""
    # Split by underscore and remove special characters
    words = "".join(c if c.isalnum() else " " for c in column).split()
    # Split camelCase
    result = []
    for word in words:
        result.extend(filter(None, re.split("([A-Z][a-z]*)", word)))
    return [w.lower() for w in result if w]

#### Implementing Jaccard Similarity

In [40]:
def jaccard_similarity(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    intersection = len(set(source_col).intersection(set(target_col)))
    union = len(set(source_col).union(set(target_col)))
    jaccard_similarity = intersection / union if union > 0 else 0
    return jaccard_similarity

#### Implementing Sequence Matcher

In [41]:
def sequence_matching(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    return SequenceMatcher(None, source_col, target_col).ratio()

#### Implementing Edit Distance

In [42]:
def editing_distance(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    max_len = max(len(source_col), len(target_col))
    return 1 - (edit_distance(source_col, target_col) / max_len)

#### Combining them all

In [43]:
def mapping_with_combination(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            final = (
                0.4 * jaccard_similarity(missing_col, extra_col)
                + 0.3 * sequence_matching(missing_col, extra_col)
                + 0.3 * editing_distance(missing_col, extra_col)
            )
            print(f"{missing_col} -> {extra_col}: {final:.2f}")
            if final > best_score:
                best_score = final
                best_match = extra_col

        if best_match and best_score > threshold:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)
    return df, missing_cols, extra_cols, mapped_cols


new_df, missing, extra_cols, mapped = mapping_with_combination(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

reserved_quantity -> reserved_stock: 0.43
reserved_quantity -> min_stock_level: 0.00
reserved_quantity -> monthly_storage_cost: 0.00
reserved_quantity -> available_qty: 0.00
reserved_quantity -> days_since_last_sale: 0.00
reserved_quantity -> stock_status: 0.00
reserved_quantity -> warehouse_location: 0.00
reserved_quantity -> total_stock_value: 0.00
reserved_quantity -> restock_lead_time_days: 0.00
reserved_quantity -> expiry_date: 0.00
reorder_level -> reserved_stock: 0.00
reorder_level -> min_stock_level: 0.32
reorder_level -> monthly_storage_cost: 0.00
reorder_level -> available_qty: 0.00
reorder_level -> days_since_last_sale: 0.00
reorder_level -> stock_status: 0.00
reorder_level -> warehouse_location: 0.00
reorder_level -> total_stock_value: 0.00
reorder_level -> restock_lead_time_days: 0.00
reorder_level -> expiry_date: 0.00
storage_cost -> reserved_stock: 0.00
storage_cost -> min_stock_level: 0.00
storage_cost -> monthly_storage_cost: 0.71
storage_cost -> available_qty: 0.00
st

#### Wordnet Mapping

In [44]:
# def preprocess_column_name(column):
#     name = re.sub("([A-Z][a-z]+)", r" \1", column)
#     name = re.sub("_", " ", name)
#     tokens = word_tokenize(name.lower())
#     return [token for token in tokens if token.isalpha()]

In [45]:
def get_wordnet_synsets(word):
    return wordnet.synsets(word)

In [46]:
def calculate_semantic_similarity(missing_col, extra_col):
    missing_tokens = preprocess_column_name(missing_col)
    extra_tokens = preprocess_column_name(extra_col)
    if not missing_tokens or not extra_tokens:
        return 0.0
    max_similarities = []
    for token1 in missing_tokens:
        synsets1 = get_wordnet_synsets(token1)
        if not synsets1:
            continue
        token_similarities = []
        for token2 in extra_tokens:
            synsets2 = get_wordnet_synsets(token2)
            if not synsets2:
                continue
            similarities = [
                s1.path_similarity(s2)
                for s1 in synsets1
                for s2 in synsets2
                if s1.path_similarity(s2) is not None
            ]
            if similarities:
                token_similarities.append(max(similarities))
        if token_similarities:
            max_similarities.append(max(token_similarities))
    return sum(max_similarities) / len(max_similarities) if max_similarities else 0.0

In [47]:
def semantic_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.6):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            similarity = calculate_semantic_similarity(missing_col, extra_col)
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col
        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)
        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)
    return df, missing_cols, extra_cols, mapped_cols
new_df, missing, extra_cols, mapped = semantic_column_mapping(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

Mapping: monthly_storage_cost -> storage_cost: 1.00

Normalized DataFrame:
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_stock|min_stock_level| last_restocked_date|      last_sold_date|storage_cost|          created_at|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|      100555| PROD_0781|    SUPP_017|            72|            21|             6

# Implementing ydata-profiling

In [48]:
# from ydata_profiling import ProfileReport
# pdf = df.toPandas()

# profile = ProfileReport(pdf, title="Pandas Profiling Report", explorative=True)
# profile.to_file("pandas_profiling_report.html")
# desc = profile.description_set

# def pandas_profiling_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.87)
#     for missing_col in missing_cols[:]:
#         best_match = None
#         best_score = 0

#         for extra_col in extra_cols[:]:
#             corr = abs(pdf[missing_col].corr(pdf[extra_col]))
#             if corr > best_score:
#                 best_score = corr
#                 best_match = extra_col

#         if best_match and best_score > threshold:
#             print(
#                 f"Data-based Mapping: {best_match} -> {missing_col} (corr={best_score:.2f})"
#             )
#             mapped_cols[missing_col] = best_match
#             pdf = pdf.rename(columns={best_match: missing_col})
#             extra_cols.remove(best_match)
#             missing_cols.remove(missing_col)
        
#     new_df = spark.createDataFrame(pdf)
#     return new_df, missing_cols, extra_cols, mapped_cols
# new_df, missing, extra_cols, mapped = pandas_profiling_mapping(
#     df, missing, extra_cols, mapped, threshold=0.87
# )

# print("\nNormalized DataFrame:")
# new_df.show(5)

# print("\nMissing columns:")
# print(missing)

# print("\nExtra columns:")
# print(extra_cols)

# print("\nMapped columns:")
# print(mapped)

# Implementing spaCy

In [49]:
import spacy

def spacy_column_mapping(
    df,
    missing_cols,
    extra_cols,
    mapped_cols,
    threshold=0.87
):
    nlp = spacy.load("en_core_web_md")

    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        missing_doc = nlp(" ".join(preprocess_column_name(missing_col)))

        for extra_col in extra_cols[:]:
            extra_doc = nlp(" ".join(preprocess_column_name(extra_col)))
            similarity = missing_doc.similarity(extra_doc)

            if similarity > best_score:
                best_score = similarity
                best_match = extra_col

        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols

new_df, missing, extra_cols, mapped = spacy_column_mapping(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)


Normalized DataFrame:
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|inventory_id|product_id| supplier_id|stock_quantity|reserved_stock|min_stock_level| last_restocked_date|      last_sold_date|storage_cost|          created_at|available_qty|days_since_last_sale|stock_status|warehouse_location|total_stock_value|restock_lead_time_days|expiry_date|
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+------------------+-----------------+----------------------+-----------+
|      100555| PROD_0781|    SUPP_017|            72|            21|             69|2025-08-25 18:33:...|2025-09-07 03:54:...|        

# Implementing Word2Vec

In [50]:
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
import re


def load_word2vec_model():
    # Get all unique column names from both dataframes
    all_columns = list(set(df.columns + extra_df.columns))
    
    # Load pre-trained Word2Vec model - you can use different pre-trained models
    try:
        # Try loading Google's pre-trained model (you need to download this separately)
        model = KeyedVectors.load_word2vec_format(
            "GoogleNews-vectors-negative300.bin", binary=True
        )
    except:
        # Fallback to training a simple model on your column names
        # This is just a basic fallback - ideally you should use a pre-trained model
        sentences = [preprocess_column_name(col) for col in all_columns]
        model = Word2Vec(sentences, vector_size=100, window=5, min_count=1)
        model = model.wv
    return model


def calculate_word2vec_similarity(col1, col2, model):
    words1 = preprocess_column_name(col1)
    words2 = preprocess_column_name(col2)

    if not words1 or not words2:
        return 0.0
    vec1 = []
    vec2 = []

    for word in words1:
        try:
            vec1.append(model[word])
        except KeyError:
            continue

    for word in words2:
        try:
            vec2.append(model[word])
        except KeyError:
            continue

    if not vec1 or not vec2:
        return 0.0

    vec1_avg = np.mean(vec1, axis=0)
    vec2_avg = np.mean(vec2, axis=0)

    similarity = np.dot(vec1_avg, vec2_avg) / (
        np.linalg.norm(vec1_avg) * np.linalg.norm(vec2_avg)
    )
    return float(similarity)


def word2vec_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    model = load_word2vec_model()

    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold

        for extra_col in extra_cols[:]:
            similarity = calculate_word2vec_similarity(missing_col, extra_col, model)
            print(f"{missing_col} -> {extra_col}: {similarity:.2f}")
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col

        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)

    return df, missing_cols, extra_cols, mapped_cols

new_df, missing, extra_cols, mapped = word2vec_column_mapping(
    df, missing, extra_cols, mapped, threshold=0.87
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

reserved_quantity -> reserved_stock: 0.56
reserved_quantity -> min_stock_level: 0.00
reserved_quantity -> available_qty: 0.10
reserved_quantity -> days_since_last_sale: 0.02
reserved_quantity -> stock_status: 0.17
reserved_quantity -> warehouse_location: -0.15
reserved_quantity -> total_stock_value: 0.02
reserved_quantity -> restock_lead_time_days: 0.01
reserved_quantity -> expiry_date: -0.01
reorder_level -> reserved_stock: -0.01
reorder_level -> min_stock_level: 0.31
reorder_level -> available_qty: 0.17
reorder_level -> days_since_last_sale: 0.12
reorder_level -> stock_status: 0.04
reorder_level -> warehouse_location: -0.06
reorder_level -> total_stock_value: -0.05
reorder_level -> restock_lead_time_days: -0.17
reorder_level -> expiry_date: 0.09

Normalized DataFrame:
+------------+----------+------------+--------------+--------------+---------------+--------------------+--------------------+------------+--------------------+-------------+--------------------+------------+-----------